# MiraeVaani 2.0 — Colab #3: TTS Service

Runs **Coqui XTTS v2** for Hindi/English TTS + **gTTS fallback** for Tamil, Telugu, Marathi, etc.

- `POST /synthesize` — XTTS v2 (Hindi, English, 15 other languages)
- `POST /synthesize_regional` — gTTS (Tamil, Telugu, Marathi, Kannada, Malayalam, etc.)
- Returns base64-encoded mulaw 8kHz audio ready for Twilio
- Exposed via ngrok

**Setup:** Runtime → Change runtime type → **T4 GPU**

In [1]:
# Cell 1: Verify GPU
import subprocess, torch
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip())
print('CUDA:', torch.cuda.is_available())
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

GPU: Tesla T4, 15360 MiB
CUDA: True
VRAM: 15.6 GB


In [2]:
# Cell 2: Install dependencies
import subprocess

subprocess.run('apt-get install -y -q ffmpeg', shell=True)

# Pin transformers BEFORE installing coqui-tts (compatibility fix)
!pip install -q "transformers==4.40.2"
!pip install -q "click==8.1.8" gtts
!pip install -q "coqui-tts @ git+https://github.com/idiap/coqui-ai-TTS.git"
!pip install -q fastapi uvicorn pyngrok httpx scipy
!pip install -q --upgrade "click>=8.4.0"

print('✅ Done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.0 requires

In [3]:
# Cell 3: Configure ngrok
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3G2uqiQ4VDdoz9pb8UgPNjgyQSr_6TpFyXshKNZQL5WSJ1HKx"  # https://dashboard.ngrok.com
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print('✅ ngrok configured')

✅ ngrok configured


In [4]:
# Cell 4: Write TTS service (with transformers compatibility patch)
tts_code = '''
import os
import base64
import logging
import tempfile
import subprocess
import audioop
import numpy as np
from scipy.signal import resample
from scipy.io import wavfile
from fastapi import FastAPI
from pydantic import BaseModel
from gtts import gTTS

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title="MiraeVaani TTS")

# --- Patch broken transformers import before loading TTS ---
import transformers.pytorch_utils as _pu
if not hasattr(_pu, "isin_mps_friendly"):
    import torch
    _pu.isin_mps_friendly = torch.isin
# --- End patch ---

from TTS.api import TTS

XTTS_LANGUAGES = {"en", "hi", "es", "fr", "de", "it", "pt", "pl", "tr",
                  "ru", "nl", "cs", "ar", "zh-cn", "ja", "hu", "ko"}

GTTS_LANG_MAP = {
    "ta": "ta", "te": "te", "mr": "mr", "bn": "bn",
    "gu": "gu", "kn": "kn", "ml": "ml", "pa": "pa",
    "ur": "ur", "or": "or", "as": "as",
}

os.environ["COQUI_TOS_AGREED"] = "1"
logger.info("Loading XTTS v2...")
tts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")
logger.info("XTTS v2 loaded")


def _to_mulaw_b64(wav_array: np.ndarray, src_rate: int) -> str:
    target = int(len(wav_array) * 8000 / src_rate)
    resampled = resample(wav_array.astype(np.float32), target)
    pcm = (np.clip(resampled, -1.0, 1.0) * 32767).astype(np.int16).tobytes()
    return base64.b64encode(audioop.lin2ulaw(pcm, 2)).decode("ascii")


class SynthRequest(BaseModel):
    text: str
    language: str = "hi"
    speaker: str = "Ana Florence"


@app.get("/health")
def health():
    return {"status": "ok", "model": "xtts-v2"}


@app.post("/synthesize")
async def synthesize(req: SynthRequest):
    lang = req.language if req.language in XTTS_LANGUAGES else "hi"
    logger.info("XTTS [%s]: %s", lang, req.text[:80])
    wav = tts_model.tts(text=req.text, language=lang, speaker=req.speaker)
    return {
        "audio_base64": _to_mulaw_b64(np.array(wav, dtype=np.float32), 24000),
        "sample_rate": 8000,
        "encoding": "mulaw",
    }


@app.post("/synthesize_regional")
async def synthesize_regional(req: SynthRequest):
    lang = GTTS_LANG_MAP.get(req.language, "hi")
    logger.info("gTTS [%s->%s]: %s", req.language, lang, req.text[:80])
    tts_obj = gTTS(text=req.text, lang=lang, slow=False)
    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as f:
        tts_obj.save(f.name)
        mp3 = f.name
    wav_path = mp3.replace(".mp3", ".wav")
    subprocess.run(["ffmpeg", "-y", "-i", mp3, "-ar", "8000", "-ac", "1", wav_path],
                   capture_output=True)
    _, data = wavfile.read(wav_path)
    os.unlink(mp3); os.unlink(wav_path)
    mulaw = audioop.lin2ulaw(data.astype(np.int16).tobytes(), 2)
    return {
        "audio_base64": base64.b64encode(mulaw).decode("ascii"),
        "sample_rate": 8000,
        "encoding": "mulaw",
    }


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8003)
'''

with open('/content/tts_service.py', 'w') as f:
    f.write(tts_code)
print('✅ tts_service.py written')

✅ tts_service.py written


In [5]:
# Cell 5: Start TTS service + expose via ngrok
import subprocess, time, httpx, os
from pyngrok import ngrok

# Kill any existing tunnels from previous runs of this cell
for t in ngrok.get_tunnels():
    print(f'Closing existing tunnel: {t.public_url}')
    ngrok.disconnect(t.public_url)

env = {**os.environ, 'COQUI_TOS_AGREED': '1'}
log = open('/content/tts.log', 'w')
proc = subprocess.Popen(['python', '/content/tts_service.py'],
                        stdout=log, stderr=log, env=env)

print('Loading XTTS v2 (~4 GB, ~2-3 min)...')
for i in range(42):
    time.sleep(5)
    if proc.poll() is not None:
        log.flush()
        print('❌ TTS service crashed. Log:')
        print(open('/content/tts.log').read()[-3000:])
        break
    try:
        r = httpx.get('http://localhost:8003/health', timeout=3)
        if r.status_code == 200:
            print(f'✅ TTS service ready ({(i+1)*5}s): {r.json()}')
            break
    except Exception:
        if i % 6 == 5:
            print(f'  Still loading... ({(i+1)*5}s)')
else:
    log.flush()
    print('⚠️  Timed out — log tail:')
    print(open('/content/tts.log').read()[-2000:])

tunnel = ngrok.connect(8003, 'http')
print()
print('=' * 60)
print('  PASTE THIS INTO YOUR LAPTOP .env')
print('=' * 60)
print(f'TTS_BASE_URL={tunnel.public_url}')
print('=' * 60)

Loading XTTS v2 (~4 GB, ~2-3 min)...
  Still loading... (30s)
✅ TTS service ready (55s): {'status': 'ok', 'model': 'xtts-v2'}

  PASTE THIS INTO YOUR LAPTOP .env
TTS_BASE_URL=https://conjoined-kettle-unmoved.ngrok-free.dev


In [6]:
# Cell 6: Quick synthesis test
import httpx, base64

print('Testing Hindi TTS...')
r = httpx.post('http://localhost:8003/synthesize',
               json={'text': 'नमस्ते, मैं वाणी बोल रही हूं मिराए एसेट शेयरखान से।', 'language': 'hi'},
               timeout=30)
audio = base64.b64decode(r.json()['audio_base64'])
print(f'✅ Hindi: {len(audio)} bytes ({len(audio)/8:.0f}ms of audio)')

print('Testing Tamil TTS (gTTS fallback)...')
r = httpx.post('http://localhost:8003/synthesize_regional',
               json={'text': 'வணக்கம், நான் வாணி பேசுகிறேன்.', 'language': 'ta'},
               timeout=30)
audio = base64.b64decode(r.json()['audio_base64'])
print(f'✅ Tamil: {len(audio)} bytes ({len(audio)/8:.0f}ms of audio)')

Testing Hindi TTS...
✅ Hindi: 45232 bytes (5654ms of audio)
Testing Tamil TTS (gTTS fallback)...
✅ Tamil: 25344 bytes (3168ms of audio)


In [ ]:
# Cell 7: Keep-alive
import time, httpx
print('TTS keep-alive running...')
i = 0
while True:
    try:
        ok = httpx.get('http://localhost:8003/health', timeout=3).status_code == 200
        i += 1
        if i % 20 == 0:
            print(f'[{i*30}s] TTS alive={ok}')
    except Exception as e:
        print(f'⚠️  {e}')
    time.sleep(30)

TTS keep-alive running...
[600s] TTS alive=True
